In [1]:
from datasets import load_dataset
from tqdm import tqdm
import torch.nn as nn
from transformers import AutoTokenizer
import torch

c:\Users\CT-PROJECT\Documents\Team10_FYP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:


dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
tokenizer_fp = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")

class Evaluator:
    def __init__(self, dataset, tokenizer, device, n_samples=5):
        self.dataset = tokenizer(
            "\n\n".join(dataset["text"]),
            return_tensors="pt"
        ).input_ids.to(device)

        self.device = device
        self.n_samples = n_samples
        self.seq_len = 512

    @torch.no_grad()
    def evaluate(self, model):
        model.eval()
        # nlls = []
        total_loss = 0
        total_tokens = 0
        loss_fct = nn.CrossEntropyLoss()

        for i in tqdm(range(self.n_samples)):
            batch = self.dataset[:, i*self.seq_len:(i+1)*self.seq_len]
            
            outputs = model(batch)
            logits = outputs.logits
            print(outputs.logits.abs().max())
            shift_logits = logits[:, :-1, :].float()
            shift_labels = batch[:, 1:]

            loss = loss_fct(
                shift_logits.reshape(-1, shift_logits.size(-1)),
                shift_labels.reshape(-1)
            )
            print("loss :",loss.item())
            num_tokens = shift_labels.numel()

            total_loss += loss.item() * num_tokens
            total_tokens += num_tokens

        ppl = torch.exp(torch.tensor(total_loss / total_tokens))
            # nlls.append(loss * self.seq_len)

        # ppl = torch.exp(torch.stack(nlls).sum() / (self.n_samples * self.seq_len))
        return ppl.item()

c:\Users\CT-PROJECT\Documents\Team10_FYP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
class Evaluator:
    def __init__(self, dataset, tokenizer, device, n_samples=40):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.device = device

        self.dataset = tokenizer(
            "\n\n".join(dataset["text"]), return_tensors="pt"
        ).input_ids.to(device)

        self.n_samples = n_samples

    @torch.no_grad()
    def evaluate(self, model):
        model.eval()
        nlls = []
        n_samples = self.n_samples if self.n_samples else self.dataset.size(1) // 2048
        for i in tqdm(range(n_samples), desc="Evaluating..."):
            batch = self.dataset[:, (i * 2048) : ((i + 1) * 2048)].to(model.device)
            with torch.no_grad():
                lm_logits = model(batch).logits
            shift_logits = lm_logits[:, :-1, :].contiguous().float()
            shift_labels = self.dataset[:, (i * 2048) : ((i + 1) * 2048)][:, 1:]
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)
            )
            neg_log_likelihood = loss.float() * 2048
            nlls.append(neg_log_likelihood)

        return torch.exp(torch.stack(nlls).sum() / (n_samples * 2048))

In [3]:
from transformers import AutoModelForCausalLM


In [ ]:
model_fp=AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf",torch_dtype=torch.float16,device_map="auto")

In [16]:
evaluator=Evaluator(dataset,tokenizer_fp,model_fp.device)

In [17]:
fp_ppl=evaluator.evaluate(model_fp)

Evaluating...: 100%|██████████| 40/40 [17:44<00:00, 26.60s/it]


In [18]:
fp_ppl

tensor(5.8230, device='cuda:0')

In [19]:
del model_fp

In [4]:
model_quantized_path='../save_dir'
model_quantized=AutoModelForCausalLM.from_pretrained(model_quantized_path,torch_dtype=torch.float16,device_map="auto")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights: 100%|██████████| 291/291 [00:33<00:00,  8.73it/s]
LlamaForCausalLM LOAD REPORT from: ../save_dir
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.layers.{0...31}.self_attn.v_proj.bias                    | UNEXPECTED |  | 
model.layers.{0...31}.self_attn.v_proj.weight_quantizer.scales | UNEXPECTED |  | 
model.layers.{0...31}.mlp.up_proj.bias                         | UNEXPECTED |  | 
model.layers.{0...31}.self_attn_q_proj_smooth_rotate           | UNEXPECTED |  | 
model.layers.{0...31}.mlp.up_proj.weight_quantizer.scales      | UNEXPECTED |  | 
model.layers.{0...31}.self_attn.k_proj.weight_quantizer.zeros  | UNEXPECTED |  | 
model.layers.{0...31}.mlp.gate_proj.bias                       | UNEXPECTED |  | 
model.layers.{0...31}.self_attn.k_proj.weight_quantizer.scales | UNEXPECTED |  | 
model.layers.{0...31}.post_attention_layernorm.bias            | U

In [22]:
del evaluator

In [11]:
tokenizer_quantized=AutoTokenizer.from_pretrained(model_quantized_path)

In [23]:
evaluator_quant=Evaluator(dataset,tokenizer_fp,model_quantized.device)

In [6]:
print(model_quantized.model.layers[0].self_attn.q_proj)

Linear(in_features=4096, out_features=4096, bias=False)


In [ ]:
quant_ppl=evaluator_quant.evaluate(model_quantized)

Evaluating...:  40%|████      | 16/40 [12:16<17:15, 43.13s/it]

In [9]:
quant_ppl

385988.03125

In [10]:
text = "Hello world"

inputs = tokenizer_fp(text, return_tensors="pt").to(model_quantized.device)

out = model_quantized(**inputs)

print(out.logits.mean())
print(out.logits.std())

tensor(0.4231, device='cuda:0', dtype=torch.float16, grad_fn=<MeanBackward0>)
tensor(2.0703, device='cuda:0', dtype=torch.float16, grad_fn=<StdBackward0>)


In [11]:
text = "The capital of France is"

inputs = tokenizer_fp(text, return_tensors="pt").to(model_quantized.device)

fp = model_fp(**inputs).logits
qt = model_quantized(**inputs).logits

print("FP logits std :", fp.std())
print("Quant logits std :", qt.std())

FP logits std : tensor(4.1523, device='cuda:0', dtype=torch.float16, grad_fn=<StdBackward0>)
Quant logits std : tensor(2.0645, device='cuda:0', dtype=torch.float16, grad_fn=<StdBackward0>)


In [22]:
layer=model_quantized.model.layers[0]


In [21]:
w

LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
    (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
    (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
)

AttributeError: 'LlamaMLP' object has no attribute 'weight_quantizer'